In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os

import numpy as np
from datasets import load_dataset
import regex as re
from tqdm import tqdm
from collections import Counter

In [3]:
def download_dataset():
    # Download a small subset
    train_ds = load_dataset(
        "roneneldan/TinyStories",
        split="train"
    )

    val_ds = load_dataset(
        "roneneldan/TinyStories",
        split="validation"
    )
    # Create Dataset Directory if not exists
    if not os.path.isdir("dataset"):
        os.mkdir("dataset")
    # Save locally as JSONL
    # train_ds.to_json(
    #     "dataset/tinystories_train.jsonl",
    #     orient="records",
    #     lines=True
    # )
    #
    # val_ds.to_json(
    #     "dataset/tinystories_val.jsonl",
    #     orient="records",
    #     lines=True
    # )

    print("Training stories:", len(train_ds))
    print("Validation stories:", len(val_ds))
    username = os.environ.get('USERNAME')
    print(f"Cache to C:\\Users\\{username}\\.cache\\huggingface\\datasets")
# download_dataset()

In [4]:
def load_tinystories():
    rt = load_dataset(
            "roneneldan/TinyStories",
            split="train",
            cache_dir="dataset",
            streaming=False
    )
    rv = load_dataset(
            "roneneldan/TinyStories",
            split="validation",
            cache_dir="dataset",
            streaming=False
    )
    return rt,rv
ds,dv = load_tinystories()

In [5]:
def save_shard_to_mmap(tokenized_data: list[list[int]], shard_prefix: str):
    """Writes a single shard to .bin and .idx files."""
    os.makedirs(os.path.dirname(shard_prefix) or ".", exist_ok=True)
    bin_path = f"{shard_prefix}.bin"
    idx_path = f"{shard_prefix}.idx"
    dtype = np.uint16  # Fits vocab_size up to 65,535

    offsets = [0]
    total_tokens = 0

    with open(bin_path, "wb") as f_bin:
        for seq in tokenized_data:
            arr = np.array(seq, dtype=dtype)
            f_bin.write(arr.tobytes())
            total_tokens += len(seq)
            offsets.append(offsets[-1] + len(seq))

    np.array(offsets, dtype=np.int64).tofile(idx_path)
    bin_size_mb = os.path.getsize(bin_path) / (1024**2)
    print(f"Saved {shard_prefix}: {len(tokenized_data):,} seqs | {total_tokens:,} tokens | {bin_size_mb:.2f} MB")


In [6]:
def export_sharded(
    dataset_train,
    dataset_val,
    tokenizer,
    shard_token_seen:int,
    max_token_seen:int,
    output_dir: str = "dataset/data_shards"
):
    os.makedirs(output_dir, exist_ok=True)

    # Handle both dict-like and method-like tokenizer EOS retrieval
    if hasattr(tokenizer, "special_tokens"):
        eos_id = tokenizer.special_tokens["<EOS>"]
        bos_id = tokenizer.special_tokens["<BOS>"]
    else:
        raise ValueError("EOS or BOS Token ID not Found.")

    # ---------------- Process Training ----------------
    print("\nProcessing Training Data...")

    tokens_seen = 0
    shard_token_count = 0
    shard_idx = 1
    current_shard : list[list[int]] = []
    pbar = tqdm(dataset_train, desc="Processing Train Data")
    for item in pbar:
        text = item["text"]
        # HuggingFace tokenizers return Encoding object; custom might return list[int]
        encoded : list[int] = tokenizer.encode(text)
        if len(encoded) <= 40 or len(encoded) > 510:
            continue
        else:
            ids = encoded
            ids.insert(0, bos_id)
            ids.append(eos_id)

            current_shard.append(ids)
            tokens_seen+=len(ids)
            shard_token_count+= len(ids)

            if shard_token_count >= shard_token_seen:
                shard_path = os.path.join(
                    output_dir,
                    f"train_{shard_idx}"
                )
                save_shard_to_mmap(
                    current_shard,
                    shard_path
                )

                current_shard.clear()
                shard_token_count = 0
                shard_idx += 1

            pbar.set_postfix({'Token Seen': tokens_seen})
            if tokens_seen>=max_token_seen:
                break
    # Save remaining stories
    if current_shard:
        shard_path = os.path.join(
            output_dir,
            f"train_{shard_idx}"
        )

        save_shard_to_mmap(
        current_shard,
        shard_path
        )

    print(f"Total tokens: {tokens_seen:,}")

    # ---------------- 2. Process Validation Shard ----------------
    print("\nProcessing Validation Data...")
    val_data = []
    for item in tqdm(dataset_val, desc="Processing Validation"):
        encoded = tokenizer.encode(item["text"])
        ids = encoded
        if len(ids) >=510:
            continue
        ids.insert(0, bos_id)
        ids.append(eos_id)
        val_data.append(ids)

    save_shard_to_mmap(val_data, os.path.join(output_dir, "val"))
    print("\nAll exported successfully.")

In [7]:
from core.tokenizer import BPE
tokenizer = BPE.load('exp_1/tokenizer','peak_base_1')

In [ ]:
export_sharded(dataset_train=ds,dataset_val=dv,tokenizer=tokenizer,output_dir="exp_1/data",shard_token_seen=44_000_000,max_token_seen=220_000_000)


Processing Training Data...


Processing Train Data:  10%|█         | 215041/2119719 [05:16<47:01, 675.08it/s, Token Seen=4.4e+7]   

Saved exp_1/data\train_1: 208,651 seqs | 44,000,116 tokens | 83.92 MB


Processing Train Data:  20%|██        | 430623/2119719 [10:43<40:59, 686.79it/s, Token Seen=8.8e+7]   

Saved exp_1/data\train_2: 209,470 seqs | 44,000,011 tokens | 83.92 MB


Processing Train Data:  30%|███       | 645812/2119719 [16:36<41:45, 588.29it/s, Token Seen=1.32e+8]  

Saved exp_1/data\train_3: 208,964 seqs | 44,000,294 tokens | 83.92 MB


Processing Train Data:  36%|███▌      | 754799/2119719 [19:57<41:08, 552.89it/s, Token Seen=1.55e+8]  